# HTR CREMMA Medieval — Expérience 3 : Arrow filtré (sans zones bruit)

**Différence vs runs précédentes :**
- Données : `train_clean.arrow` / `dev_clean.arrow` — 1 024 lignes bruit retirées (MusicZone, DropCapital…)
- Mode L (grayscale) confirmé
- 18 769 lignes train / 3 702 lignes dev

**Prérequis :**
- Kaggle Secrets : `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY`
- Accelerator : **GPU T4 x2**

**Ordre d'exécution :** 1 → 2 → 6a → 6b → 7  
> Ne pas exécuter les cellules 3, 4, 5 (inutiles avec le workflow Arrow)

In [ ]:
# Cellule 1 — Connexion S3
from kaggle_secrets import UserSecretsClient
import boto3
import os

secrets = UserSecretsClient()
s3 = boto3.client(
    's3',
    aws_access_key_id=secrets.get_secret('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=secrets.get_secret('AWS_SECRET_ACCESS_KEY'),
    region_name='eu-west-3'
)
print('Connexion S3 OK')

In [ ]:
# Cellule 2 — Installer Kraken
import subprocess, sys, torch

print(f'PyTorch existant : {torch.__version__}')
print(f'CUDA dispo : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU : {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})')
    if cap[0] >= 8:
        print('bf16 supporte (Ampere+)')
    else:
        print('fp16 uniquement (pas de bf16 sur cette architecture)')

TORCH_VER = torch.__version__

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kraken',
    'iso639-lang',
    f'torch=={TORCH_VER}',
], check=True)

print('\n--- Test des imports Kraken ---')
test = subprocess.run(
    [sys.executable, '-c',
     'from kraken.lib.xml import XMLPage; '
     'from kraken.train import KrakenTrainer; '
     'print("Imports OK")'],
    capture_output=True, text=True
)
print(test.stdout)
if test.returncode != 0:
    print('ERREUR IMPORT :')
    print(test.stderr[-1500:])

for pkg in ['kraken', 'torch', 'lightning', 'torchmetrics']:
    v = subprocess.run([sys.executable, '-c', f'import importlib.metadata as m; print(m.version("{pkg}"))'],
                       capture_output=True, text=True)
    print(f'{pkg}: {v.stdout.strip() or v.stderr.strip()}')

In [ ]:
# Cellule 6a — Telecharger les Arrow filtres (Exp 3) + modele de base depuis S3
import os

os.makedirs('/kaggle/working/splits', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)

for s3_key, local in [
    ('splits/train_clean.arrow',                '/kaggle/working/splits/train_clean.arrow'),
    ('splits/dev_clean.arrow',                  '/kaggle/working/splits/dev_clean.arrow'),
    ('base-model/cremma-generic-1.0.1.mlmodel', '/kaggle/working/cremma_generic.mlmodel'),
]:
    print(f'Telechargement {s3_key}...')
    s3.download_file('htr-cremma-medieval', s3_key, local)
    size = os.path.getsize(local) / 1024 / 1024
    print(f'OK : {local} ({size:.1f} MB)')

# Manifests Arrow
with open('/kaggle/working/splits/train_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/train_clean.arrow\n')
with open('/kaggle/working/splits/dev_binary.txt', 'w') as f:
    f.write('/kaggle/working/splits/dev_clean.arrow\n')

print('Manifests OK')
print('train_clean.arrow : 18 769 lignes (1 024 zones bruit retirees)')
print('dev_clean.arrow   : 3 702 lignes')

In [ ]:
# Cellule 6b — Fine-tuning Exp 3 (Arrow filtre, grayscale, T4)
import subprocess, os, time, datetime, re

cmd = [
    'ketos',
    '--device', 'cuda:0',
    '--workers', '4',
    '--precision', '16-mixed',
    '--threads', '4',
    'train',
    '-f', 'binary',
    '-t', '/kaggle/working/splits/train_binary.txt',
    '-e', '/kaggle/working/splits/dev_binary.txt',
    '-i', '/kaggle/working/cremma_generic.mlmodel',
    '--resize', 'union',
    '-o', '/kaggle/working/models/htr_cremma_exp3',
    '-q', 'early',
    '-N', '50',
    '--lag', '10',
    '--min-epochs', '3',
    '-B', '8',
    '-r', '0.0001',
    '--augment',
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TERM'] = 'dumb'

print('Commande :', ' '.join(cmd))
print(f'Debut    : {datetime.datetime.now().strftime("%H:%M:%S")}')
print('-' * 60)

t_global = time.time()
t_epoch = time.time()
epoch_num = 0
best_acc = 0.0
best_epoch = 0

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=0, env=env)
buf = ''
while True:
    ch = proc.stdout.read(1)
    if not ch:
        break
    if ch == '\r':
        print('\r' + buf, end='', flush=True)
        buf = ''
    elif ch == '\n':
        line = buf
        print(line)

        m = re.search(r'stage\s+(\d+)', line, re.IGNORECASE)
        if m:
            epoch_num = int(m.group(1))
            t_epoch = time.time()

        m_acc = re.search(r'val_accuracy[:\s]+([\d.]+)', line, re.IGNORECASE)
        if m_acc:
            acc = float(m_acc.group(1))
            elapsed_epoch = time.time() - t_epoch
            elapsed_total = time.time() - t_global
            if acc > best_acc:
                best_acc = acc
                best_epoch = epoch_num
            print(f'  [Epoch {epoch_num:>2}] val_accuracy={acc*100:.2f}% | CER={((1-acc)*100):.2f}% | '
                  f'duree={elapsed_epoch:.0f}s | total={elapsed_total/60:.1f}min')
            print(f'  Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')

        m_pat = re.search(r'patience\s+(\d+)/(\d+)', line, re.IGNORECASE)
        if m_pat:
            cur, total_pat = int(m_pat.group(1)), int(m_pat.group(2))
            print(f'  Patience : {cur}/{total_pat} — early stop dans {total_pat - cur} epoch(s)')

        buf = ''
    else:
        buf += ch

if buf:
    print(buf)
proc.wait()

elapsed_total = time.time() - t_global
print('-' * 60)
print(f'Fin      : {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'Duree totale : {elapsed_total/60:.1f} min')
print(f'Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% acc / {(1-best_acc)*100:.2f}% CER')
print(f'Code retour : {proc.returncode}')

In [ ]:
# Cellule 7 — Uploader le modele final sur S3
import glob, datetime
from pathlib import Path

models_dir = '/kaggle/working/models'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')

mlmodels    = glob.glob(f'{models_dir}/**/*.mlmodel', recursive=True)
safetensors = glob.glob(f'{models_dir}/**/*.safetensors', recursive=True)
checkpoints = glob.glob(f'{models_dir}/**/*.ckpt', recursive=True)

print('Fichiers trouves :')
print(f'  .mlmodel     : {mlmodels}')
print(f'  .safetensors : {safetensors}')
print(f'  .ckpt        : {checkpoints}')

uploaded = []

for f in mlmodels:
    nom = f'exp3_clean_arrow_{timestamp}.mlmodel'
    s3.upload_file(f, 'htr-cremma-medieval', f'output/{nom}')
    print(f'Upload : s3://htr-cremma-medieval/output/{nom}')
    uploaded.append(nom)

if not mlmodels:
    for f in safetensors:
        nom = f'exp3_clean_arrow_{timestamp}.safetensors'
        s3.upload_file(f, 'htr-cremma-medieval', f'output/{nom}')
        print(f'Upload : s3://htr-cremma-medieval/output/{nom}')
        uploaded.append(nom)

if not mlmodels and not safetensors and checkpoints:
    best = sorted(checkpoints, key=lambda p: float(Path(p).stem.split('-')[-1]) if '-' in Path(p).stem else 0, reverse=True)[0]
    nom = f'exp3_clean_arrow_{timestamp}.ckpt'
    s3.upload_file(best, 'htr-cremma-medieval', f'output/{nom}')
    print(f'Upload (checkpoint) : s3://htr-cremma-medieval/output/{nom}')
    uploaded.append(nom)

if not uploaded:
    print('ERREUR : aucun modele trouve dans', models_dir)
    for f in Path(models_dir).rglob('*'):
        print(f'  {f}')
else:
    print(f'OK — {len(uploaded)} fichier(s) uploade(s) sur S3')